# Extract Transcripts from PDF Files

Reads all transcript PDFs and extracts timestamp, speaker, and text into a single CSV.

**PDF format:**
```
HH:MM:SS:FF   SPEAKER   Text of utterance...
```

**Speaker codes:** `T` (Teacher), `SN` (Student-N/group), `S` (individual Student), `S?` (unidentified), `Ss` (multiple students), `O` (other)

In [ ]:
import pdfplumber
import pandas as pd
import re
from pathlib import Path

TRANSCRIPT_DIR = Path("../transcripts")
OUTPUT_CSV = Path("transcripts.csv")  # saved in preprocessing/

# Timestamp: HH:MM:SS:FF  (frames, not milliseconds)
TIMESTAMP_RE = re.compile(r'^\d{2}:\d{2}:\d{2}:\d{2}$')

# Known speaker codes found in corpus:
# T=Teacher, SN=Student-group, S=single Student, Ss/SS=multiple students,
# S?=unidentified student, O=Other/observer, NS=non-speaker, E=expert
SPEAKER_RE = re.compile(r'^(T|SN|Ss|SS|NS|S\??|O|E|L|LN|LS|[A-Z]{1,3}\d*)$')

pdf_files = sorted(TRANSCRIPT_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files")

In [2]:
def parse_pdf(pdf_path: Path) -> list[dict]:
    """
    Extract all utterances from a single PDF.
    Returns a list of dicts with keys: filename, timestamp, speaker, text.
    """
    rows = []
    filename = pdf_path.name

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # Extract words with bounding box positions
            words = page.extract_words(x_tolerance=4, y_tolerance=4)
            if not words:
                continue

            # Group words into lines by y-coordinate (round to nearest 3 px)
            lines: dict[int, list] = {}
            for word in words:
                y_key = round(word["top"] / 3) * 3
                lines.setdefault(y_key, []).append(word)

            # Determine x-boundary between timestamp and speaker columns.
            # The timestamp column ends roughly where the speaker codes start.
            # We detect this dynamically from lines that have a timestamp.
            ts_lines = [
                sorted(words_in_line, key=lambda w: w["x0"])
                for words_in_line in lines.values()
                if words_in_line and TIMESTAMP_RE.match(
                    sorted(words_in_line, key=lambda w: w["x0"])[0]["text"]
                )
            ]

            # Process sorted lines top-to-bottom
            for y_key in sorted(lines.keys()):
                line_words = sorted(lines[y_key], key=lambda w: w["x0"])
                parts = [w["text"] for w in line_words]

                if not parts:
                    continue

                # Skip page number lines (single token that is just a number or "- N -")
                if len(parts) <= 3 and all(p in ["-"] or p.isdigit() for p in parts):
                    continue

                if TIMESTAMP_RE.match(parts[0]):
                    # New utterance line
                    timestamp = parts[0]
                    rest = parts[1:]

                    # Speaker is the next token if it matches the speaker pattern
                    if rest and SPEAKER_RE.match(rest[0]):
                        speaker = rest[0]
                        text = " ".join(rest[1:])
                    else:
                        speaker = ""
                        text = " ".join(rest)

                    rows.append({
                        "filename": filename,
                        "timestamp": timestamp,
                        "speaker": speaker,
                        "text": text.strip(),
                    })

                elif rows and rows[-1]["filename"] == filename:
                    # Continuation of the previous utterance (no timestamp)
                    continuation = " ".join(parts).strip()
                    # Skip pure page-header lines like "- 2 -"
                    if re.fullmatch(r'-\s*\d+\s*-', continuation):
                        continue
                    rows[-1]["text"] = (rows[-1]["text"] + " " + continuation).strip()

    return rows

In [3]:
all_rows = []
errors = []

for i, pdf_path in enumerate(pdf_files):
    try:
        rows = parse_pdf(pdf_path)
        all_rows.extend(rows)
        if (i + 1) % 20 == 0 or i == len(pdf_files) - 1:
            print(f"  [{i+1}/{len(pdf_files)}] {pdf_path.name}  →  {len(rows)} utterances")
    except Exception as e:
        errors.append((pdf_path.name, str(e)))
        print(f"  ERROR in {pdf_path.name}: {e}")

print(f"\nTotal utterances extracted: {len(all_rows)}")
if errors:
    print(f"Files with errors: {[e[0] for e in errors]}")

  [20/141] T-1109-Lek2-Transkr-Video.pdf  →  607 utterances
  [40/141] T-1118-Tutor-1-1-Transkr-Video.pdf  →  130 utterances
  [60/141] T-1218-Lek2-Transkr-Video.pdf  →  534 utterances
  [80/141] T-2102-Tutor-1-4-Transkr-Video.pdf  →  147 utterances
  [100/141] T-2108-Tutor-1-4-Transkr-Video.pdf  →  122 utterances
  [120/141] T-2114-Tutor-1-1-Transkr-Video.pdf  →  222 utterances
  [140/141] T-2205-Tutor-1-1-Transkr-Video.pdf  →  368 utterances
  [141/141] T-2205-Tutor-1-4-Transkr-Video.pdf  →  318 utterances

Total utterances extracted: 48605


In [4]:
df = pd.DataFrame(all_rows, columns=["filename", "timestamp", "speaker", "text"])

# Quick sanity check
print(f"Shape: {df.shape}")
print(f"\nSpeaker distribution:")
print(df["speaker"].value_counts().to_string())
print(f"\nFiles processed: {df['filename'].nunique()}")
df.head(10)

Shape: (48605, 4)

Speaker distribution:
speaker
T     26247
SN    10709
S      9915
S?      662
Ss      528
NS      230
O       186
E        58
         40
SS       30

Files processed: 141


,filename,timestamp,speaker,text
0,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:01:06,T,"Da steht jetzt keine Frage bei, (was sollen wi..."
1,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:04:27,SN,( ).
2,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:06:19,T,( ).
3,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:12:17,S,"Ja, wieviele Tiere da drin- da drin sind. Wiev..."
4,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:19:14,T,Mhm. Kannst du denn die eventuell schon beantw...
5,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:23:03,S,(Weinbergschnecken haben keine Füsse).
6,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:25:27,T,{Lacht} Ok. ( ). Und Beine haben sie auch keine.
7,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:35:20,S,"Ja, ( ). Da hätte ich schon mal fünfunddreissi..."
8,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:41:08,T,// Mhm. Genau.
9,T-1101-Tutor-1-1-Transkr-Video.pdf,00:00:43:25,T,Kannst du ja schon mal machen.


In [ ]:
# ── Speaker normalisation ─────────────────────────────────────────────────
# Collapse all student variants (SN, Ss, SS, S?) into a single "S" label
STUDENT_VARIANTS = {"SN", "Ss", "SS", "S?"}
df["speaker"] = df["speaker"].apply(lambda s: "S" if s in STUDENT_VARIANTS else s)

print("Updated speaker distribution:")
print(df["speaker"].value_counts().to_string())

# ── Text cleaning ─────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return text
    # Remove non-verbal / paralinguistic annotations: {Lacht}, {Lärm}, …
    text = re.sub(r'\{[^}]*\}', '', text)
    # Remove transcriber notes in square brackets: [Ja], [ironisch], …
    text = re.sub(r'\[[^\]]*\]', '', text)
    # Remove simultaneous-speech markers: //, ///, etc.
    text = re.sub(r'\s*/+\s*', ' ', text)
    # Normalize curly apostrophes to straight apostrophe
    text = re.sub(u'[\u2018\u2019]', "'", text)
    # Remove emphasis asterisks, keep content: *wir* → wir
    text = re.sub(r'\*([^*]*)\*', r'\1', text)
    # Remove all double/curly/angle quote characters
    text = re.sub(u'[\u201c\u201d\u201e\u00ab\u00bb"]', '', text)
    # Remove empty parentheses first: ( ), ()
    text = re.sub(r'\(\s*\)', '', text)
    # Strip parentheses but keep content: (text) → text
    text = re.sub(r'\(([^)]+)\)', lambda m: m.group(1).strip(), text)
    # Clean up punctuation artifacts left by removal
    text = re.sub(r'\s+([.,;:?!])', r'\1', text)    # "word ." → "word."
    text = re.sub(r'([.!?])\s*[.!?]+', r'\1', text)  # ". ." → "."
    text = re.sub(r'[,;]\s*\.', '.', text)           # ",." → "."
    # Collapse multiple whitespace to a single space
    text = re.sub(r'[ \t]+', ' ', text)
    text = text.strip()
    # Return empty string if only punctuation/whitespace remains
    if re.fullmatch(r'[.,;:?!\s…\.]*', text):
        return ''
    return text

df["text"] = df["text"].apply(clean_text)

print("\nSample cleaned rows:")
df[["speaker", "text"]].head(10)

In [5]:
# Show any rows with an empty or unexpected speaker (may need to extend SPEAKER_RE)
unexpected = df[df["speaker"] == ""]
print(f"Rows with no detected speaker: {len(unexpected)}")
if len(unexpected) > 0:
    print("\nSample rows with missing speaker (first 10):")
    display(unexpected.head(10))

Rows with no detected speaker: 40

Sample rows with missing speaker (first 10):


,filename,timestamp,speaker,text
593,T-1103-Lek1-Transkr-Video.pdf,00:39:01:19,,( )
746,T-1103-Lek1-Transkr-Video.pdf,00:48:28:20,,Bitte?
2701,T-1104-Tutor-1-4-Transkr-Video.pdf,00:17:26:11,,"Noch // eine Stunde brennen, ja. Und dann wäre..."
4095,T-1106-Tutor-1-1-Transkr-Video.pdf,00:01:32:25,,// Okay. Ja.
4216,T-1106-Tutor-1-1-Transkr-Video.pdf,00:07:31:10,,{ Lärm hinter dem Lehrer}
4622,T-1106-Tutor-1-4-Transkr-Video.pdf,00:08:14:23,,Sn // Es stimmt
4758,T-1106-Tutor-1-4-Transkr-Video.pdf,00:13:05:20,,O? ( )
9410,T-1110-Tutor-1-4-Transkr-Video.pdf,00:12:59:19,,Ja.
12433,T-1117-Lek2-Transkr-Video.pdf,00:50:20:14,,O? ()
12978,T-1118-Lek1-Transkr-Video.pdf,00:10:20:15,,Sn // ( )


In [6]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows to: {OUTPUT_CSV.resolve()}")

Saved 48605 rows to: /Users/emilbreustedt/Documents/Arbeit/HIEB/Multi-Stage Speaker Diarization/transcripts.csv
